In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/.gitignore
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/readme.md
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/requirements.txt
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/.env
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/.env.example
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/pipeline_paper.png
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/BEA_camera_ready.pdf
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/dt_code/run_baseline.py
/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/dt_code/tutor

In [2]:
!cp -r "/kaggle/input/datasets/chakrounoussama/multi-v3/Multi-Agent-Tutoring-Systems" /kaggle/working/

In [3]:
%cd /kaggle/working

/kaggle/working


In [4]:
%cd /kaggle/working/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/dt_code

/kaggle/working/Multi-Agent-Tutoring-Systems/Multi-Agent-Tutoring-Systems/dt_code


In [5]:
with open("run_graph_pipeline.py", "r") as f:
    content = f.read()

content = content.replace(
    '_PROVIDER_CHOICES = ["mistral", "groq", "gemini"]',
    '_PROVIDER_CHOICES = ["mistral", "groq", "gemini", "local"]'
)

with open("run_graph_pipeline.py", "w") as f:
    f.write(content)

print("patched" if '"local"' in content.split("_PROVIDER_CHOICES")[1][:60] else "NO MATCH FOUND — check the line manually")

patched


In [6]:
with open("llm_client.py", "r") as f:
    content = f.read()

changed = False

# 1. Add the DEFAULT_LOCAL_MODEL constant if missing
if "DEFAULT_LOCAL_MODEL" not in content:
    old_const = 'DEFAULT_MISTRAL_MODEL = "mistral-large-latest"'
    new_const = old_const + '\n\nDEFAULT_LOCAL_MODEL = "Qwen/Qwen2.5-7B-Instruct"'
    if old_const in content:
        content = content.replace(old_const, new_const)
        changed = True
        print("Added DEFAULT_LOCAL_MODEL constant.")
    else:
        print("WARNING: could not find DEFAULT_MISTRAL_MODEL line to anchor the constant insert.")

# 2. Add the local-model loader + _call_local function if missing
if "_call_local" not in content:
    local_block = '''

# --------------------------------------------------------------------------
# Local provider -- runs an open model in-process via `transformers`, on
# whatever GPU/CPU is available. No API key, no quota, no rate limit.
# --------------------------------------------------------------------------

_LOCAL_MODELS = {}          # model_name -> (tokenizer, model)
_LOCAL_LOCK = threading.Lock()


def _load_local_model(model_name, load_in_4bit=None):
    """Lazily load + cache a HF model/tokenizer. Reused across all calls in
    the process so the (slow) load only happens once per model name."""
    with _LOCAL_LOCK:
        if model_name in _LOCAL_MODELS:
            return _LOCAL_MODELS[model_name]

        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
        except ImportError as e:
            raise LLMError(
                "Local provider requires `torch` and `transformers`. Install with:\\n"
                "  pip install torch transformers accelerate bitsandbytes\\n"
                f"(original error: {e})"
            )

        if load_in_4bit is None:
            load_in_4bit = False
            if torch.cuda.is_available():
                total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
                load_in_4bit = total_gb < 20

        print(f"  [local] loading {model_name} "
              f"({'4-bit' if load_in_4bit else 'fp16/bf16' if torch.cuda.is_available() else 'cpu'})"
              f" -- first call only, this can take a couple of minutes...")

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        model_kwargs = {"device_map": "auto"}
        if load_in_4bit:
            try:
                from transformers import BitsAndBytesConfig
                model_kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_quant_type="nf4",
                )
            except ImportError:
                raise LLMError(
                    "4-bit loading requires `bitsandbytes`. Install with: "
                    "pip install bitsandbytes"
                )
        elif torch.cuda.is_available():
            model_kwargs["torch_dtype"] = torch.bfloat16

        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        model.eval()

        _LOCAL_MODELS[model_name] = (tokenizer, model)
        return tokenizer, model


def _call_local(prompt, model=DEFAULT_LOCAL_MODEL, temperature=0.2, max_tokens=2048):
    import torch

    tokenizer, hf_model = _load_local_model(model)

    messages = [{"role": "user", "content": prompt}]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    encoded = {k: v.to(hf_model.device) for k, v in encoded.items()}
    input_len = encoded["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = hf_model.generate(
            **encoded,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if not text.strip():
        raise LLMError("Local model returned empty content.")
    return text

'''
    anchor = 'def call_llm(prompt, provider="groq"'
    if anchor in content:
        content = content.replace(anchor, local_block.strip("\n") + "\n\n\n" + anchor)
        changed = True
        print("Added _load_local_model / _call_local functions.")
    else:
        print("WARNING: could not find call_llm() to anchor the function insert.")

# 3. Wire "local" into call_llm()'s dispatcher if missing
if 'elif provider == "local"' not in content:
    old_dispatch = '''    elif provider == "mistral":
        return _call_mistral(prompt, model=model or DEFAULT_MISTRAL_MODEL,
                             temperature=temperature, max_tokens=max_tokens)
    else:
        raise ValueError(f"Unknown provider: {provider}")'''
    new_dispatch = '''    elif provider == "mistral":
        return _call_mistral(prompt, model=model or DEFAULT_MISTRAL_MODEL,
                             temperature=temperature, max_tokens=max_tokens)
    elif provider == "local":
        return _call_local(prompt, model=model or DEFAULT_LOCAL_MODEL,
                            temperature=temperature, max_tokens=max_tokens)
    else:
        raise ValueError(f"Unknown provider: {provider}")'''
    if old_dispatch in content:
        content = content.replace(old_dispatch, new_dispatch)
        changed = True
        print("Wired 'local' into call_llm() dispatcher.")
    else:
        print("WARNING: mistral elif block didn't match exactly -- paste your actual call_llm() body and I'll fix the match.")

if changed:
    with open("llm_client.py", "w") as f:
        f.write(content)
    print("\nFile saved.")
else:
    print("\nNothing changed -- check warnings above.")


Nothing changed -- check warnings above.


In [7]:
!rm -rf /kaggle/working/Multi-Agent-Tutoring-Systems/dt_code/Data/llm_output/pipeline_run.jsonl


In [8]:
!pip install -U bitsandbytes>=0.46.1

In [9]:
!python run_graph_pipeline.py --n 1728 --seed 42 \
  --student-provider local --student-model Qwen/Qwen2.5-7B-Instruct \
  --tutor-provider local --tutor-model Qwen/Qwen2.5-7B-Instruct \
  --verifier-provider local --verifier-model Qwen/Qwen2.5-7B-Instruct \
  --recovery-provider local --recovery-model Qwen/Qwen2.5-7B-Instruct \
  --sleep 0

Loaded 516 proof states total.
Sampled 516 states across 32 problems.
Orchestration: LangGraph tutoring core (Student + KG outside)
Student model:   local/Qwen/Qwen2.5-7B-Instruct (temp=0.7)
Tutor model:     local/Qwen/Qwen2.5-7B-Instruct (temp=0.2)
Verifier model:  local/Qwen/Qwen2.5-7B-Instruct (temp=0.2)
Recovery model:  local/Qwen/Qwen2.5-7B-Instruct (temp=0.2)
[1/516] id=195 problem=4.3
  [local] loading Qwen/Qwen2.5-7B-Instruct (4-bit) -- first call only, this can take a couple of minutes...
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.29MB/s]
tokenizer_config.json: 7.30kB [00:00, 18.6MB/s]
vocab.json: 2.78MB [00:00, 81.5MB/s]
merges.txt: 1.67MB [00:00, 105MB/s]
tokenizer.json: 7.03MB [00:00, 117MB/s]
model.safetensors.index.json: 27.8kB [00:00, 85.5MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:01<00:00, 15.47s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:01<00:00, 246MB/s]
Loading weights: 100%|█| 339/339 [00:1